# AutoGluon Time Series — `target_3ay_hiz`

Bu notebook, üç aylık toplam otomobil devir hızını doğrudan tahmin eden nihai doğrulama çalışmasını yeniden üretir.

- AutoGluon ayarı: `medium_quality`
- Tahmin ufku: 3 ay
- Puanlanan çıktı: yalnız üçüncü adımın (`h=3`) medyanı
- Doğrulama originleri: 2020-04 – 2025-03, toplam 60 ay
- Test originleri: 2025-04 – 2026-03; **bu notebookta açılmaz**

Uzun eğitim checkpointlidir. Mevcut sonuçları incelemek için yeniden eğitim gerekmez.


In [1]:
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Notebook notebooks/ klasöründen veya proje kökünden açılabilir.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "data").exists(), "Proje kökü bulunamadı."

try:
    import autogluon.timeseries as agts
except ImportError as exc:
    raise RuntimeError(
        "Bu notebook'u projenin .venv-ag Python ortamıyla çalıştırın."
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Proje:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("AutoGluon:", agts.__version__)


Proje: C:\Users\YOGA\Desktop\araç piyasasında yön analizi için kullanılan yöntemler v2
Python: 3.12.7
AutoGluon: 1.6.1


## Targetın anlamı

Karar ayı `t` olmak üzere:

\[
y_t = 100\ln\left(
\frac{V_{t+1}+V_{t+2}+V_{t+3}}
{V_{t-2}+V_{t-1}+V_t}
\right)
\]

- Pozitif değer: gelecek üç aylık toplam hacim, son üç aylık toplamdan yüksek.
- Negatif değer: gelecek üç aylık toplam hacim daha düşük.
- Mutlak değer: logaritmik değişimin şiddeti.

Veri setindeki target gerçekleşme ayına yazılmıştır. AutoGluon `prediction_length=3` ile bu serinin üç adım sonrasını tahmin eder; yalnız `h=3` sonucu iş targetına karşılık gelir.


In [2]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "birlesik_target_setleri"
    / "target_3ay_hiz_tum_featurelar_final.csv"
)

data = pd.read_csv(DATA_PATH)
data["referans_ayi"] = pd.to_datetime(data["referans_ayi"], errors="raise")
data = data.sort_values("referans_ayi").reset_index(drop=True)

expected_months = pd.date_range(
    data["referans_ayi"].min(), data["referans_ayi"].max(), freq="MS"
)

assert data.shape == (97, 13)
assert data["referans_ayi"].equals(pd.Series(expected_months, name="referans_ayi"))
assert not data["referans_ayi"].duplicated().any()
assert data["target_3ay_hiz"].notna().all()
assert np.isfinite(data["target_3ay_hiz"]).all()

print("Tarih:", data["referans_ayi"].min().date(), "→", data["referans_ayi"].max().date())
print("Gözlem:", len(data))
display(data.head(3))


Tarih: 2018-06-01 → 2026-06-01
Gözlem: 97


,referans_ayi,arabam_ortalama_ilan_fiyati_tl,betam_dom_gun,betam_talep_aylik_pct,otv_event_ay_mi,betam_satis_orani_pct,indicata_satisa_donen_adet,osd_binek_adet,indicata_satis_ilan_orani_pct,arabam_reel_aylik_degisim_pct,indicata_perakende_fiyat_aylik_pct,noter_devir_otomobil_adet,target_3ay_hiz
0,2018-06-01,NaN,NaN,NaN,0,NaN,NaN,"85,635.0000",NaN,NaN,NaN,"443,151.0000",3.8019
1,2018-07-01,NaN,NaN,NaN,0,NaN,NaN,"96,511.0000",NaN,NaN,NaN,"541,226.0000",5.5613
2,2018-08-01,NaN,NaN,NaN,0,NaN,NaN,"29,289.0000",NaN,NaN,NaN,"429,606.0000",-1.1957


## Target formülünü veriden doğrulama

Dosyanın ilk beş satırında önceki üç aylık blok dosya dışında kaldığı için karşılaştırma altıncı satırdan itibaren yapılabilir.


In [3]:
volume = data["noter_devir_otomobil_adet"]
rolling_3m = volume.rolling(3).sum()
reconstructed = 100 * np.log(rolling_3m / rolling_3m.shift(3))

check_mask = reconstructed.notna()
max_target_difference = (
    reconstructed[check_mask] - data.loc[check_mask, "target_3ay_hiz"]
).abs().max()

assert max_target_difference < 1e-9
print("Formül doğrulandı. En büyük fark:", max_target_difference)


Formül doğrulandı. En büyük fark: 7.105427357601002e-15


## Deney kolları ve sızıntı koruması

- **T0:** yalnız target geçmişi.
- **T1:** noter otomobil devri, OSD binek, ÖTV olayı; tamamı `lag3`.

`lag3`, hedef ayındaki kovaryant değerinin üç ay önceki kaynaktan gelmesini sağlar. Böylece `t+1`, `t+2` ve `t+3` için verilen değerlerin kaynak tarihleri karar ayı `t` veya daha erkendir.


In [4]:
T1_FEATURES = [
    "noter_devir_otomobil_adet",
    "osd_binek_adet",
    "otv_event_ay_mi",
]

source = data.set_index("referans_ayi")[T1_FEATURES]
lagged = source.shift(3)
lagged.columns = [f"{column}_lag3" for column in lagged.columns]

sample_origin = pd.Timestamp("2024-04-01")
future_months = pd.date_range(sample_origin + pd.DateOffset(months=1), periods=3, freq="MS")

for future_month in future_months:
    source_month = future_month - pd.DateOffset(months=3)
    assert source_month <= sample_origin
    print(f"Tahmin ayı {future_month:%Y-%m} ← kaynak {source_month:%Y-%m}")


Tahmin ayı 2024-05 ← kaynak 2024-02
Tahmin ayı 2024-06 ← kaynak 2024-03
Tahmin ayı 2024-07 ← kaynak 2024-04


## Eğitimi çalıştırma veya mevcut checkpointi kullanma

`RUN_TRAINING=False` mevcut tamamlanmış sonuçları yükler. Baştan eğitim için `True` yapın. Eğitim yarıda kesilirse `RESUME=True` ile tamamlanan originler atlanır.


In [5]:
def load_module(module_name: str, script_path: Path):
    """Bir Python betiğini fonksiyonlarına erişebileceğimiz modül olarak yükler."""
    spec = importlib.util.spec_from_file_location(module_name, script_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


RUN_TRAINING = False
RESUME = True

runner = load_module(
    "ag_target_3ay_runner",
    PROJECT_ROOT / "scripts" / "ag_04_uzun_pencere_h3.py",
)

if RUN_TRAINING:
    model_data = runner.load_data()
    predictions = runner.run(model_data, resume=RESUME)
    scored = runner.score(predictions)
    ranking = runner.make_ranking(scored)
    paired, mcnemar = runner.comparisons(scored)
    reproduction = runner.reproduction(predictions)
    verdict = runner.decision(ranking, paired)

    ranking.to_csv(runner.RANK_PATH, index=False, encoding="utf-8-sig")
    paired.to_csv(runner.PAIRED_PATH, index=False, encoding="utf-8-sig")
    mcnemar.to_csv(runner.MCNEMAR_PATH, index=False, encoding="utf-8-sig")
    runner.REPRO_PATH.write_text(reproduction, encoding="utf-8")
    runner.DECISION_PATH.write_text(verdict, encoding="utf-8")
else:
    predictions = pd.read_csv(runner.PRED_PATH, parse_dates=["origin", "hedef_ay"])
    ranking = pd.read_csv(runner.RANK_PATH)
    paired = pd.read_csv(runner.PAIRED_PATH)
    mcnemar = pd.read_csv(runner.MCNEMAR_PATH)
    verdict = runner.DECISION_PATH.read_text(encoding="utf-8")

print("Tahmin satırı:", len(predictions))


Tahmin satırı: 1200


## Güvenlik ve bütünlük kontrolleri


In [6]:
assert predictions["origin"].min() == pd.Timestamp("2020-04-01")
assert predictions["origin"].max() == pd.Timestamp("2025-03-01")
assert predictions["origin"].nunique() == 60
assert len(predictions[predictions["origin"].between("2025-04-01", "2026-03-01")]) == 0
assert predictions["y_pred_q50"].notna().all()
assert np.isfinite(predictions["y_pred_q50"]).all()

print("Test origin satırı: 0")
print("Tahminler sonlu ve eksiksiz.")


Test origin satırı: 0
Tahminler sonlu ve eksiksiz.


## Sonuç tabloları


In [7]:
important_models = ["Chronos2", "Toto2", "DirectTabular", "sifir"]
summary = ranking[
    ranking["rejim"].eq("TUM") & ranking["model"].isin(important_models)
].sort_values("MAE")

display(summary)
display(paired)
print(verdict)


,rejim,kol,model,n,MAE,RMSE,DA_adet,DA_yuzde
40,TUM,T1,Chronos2,60,19.8257,26.5594,38,63.3333
41,TUM,T0,sifir,60,20.0503,25.0764,0,0.0000
42,TUM,T1,sifir,60,20.0503,25.0764,0,0.0000
43,TUM,T1,DirectTabular,60,21.0799,26.0011,35,58.3333
44,TUM,T0,DirectTabular,60,21.6053,27.8242,36,60.0000
45,TUM,T0,Toto2,60,22.3529,31.0514,36,60.0000
46,TUM,T1,Toto2,60,22.3529,31.0514,36,60.0000
50,TUM,T0,Chronos2,60,25.5800,34.2051,27,45.0000


,rejim,karsilastirma,n,paired_win,paired_tie,candidate_MAE,reference_MAE,reference_eksi_candidate_MAE,net_hata_kazanci,en_buyuk_2_kazanc_cikarilinca_net_fark
0,SOK_2020_04_2022_12,ABLASYON_T1_CHRONOS2_vs_T0_CHRONOS2,33,25,0,26.2254,35.5848,9.3594,308.8600,233.4761
1,SOK_2020_04_2022_12,IS_ESIGI_T1_CHRONOS2_vs_SIFIR,33,18,0,26.2254,26.9540,0.7286,24.0450,-8.6034
2,SOK_2020_04_2022_12,KONTROL_T1_CHRONOS2_vs_TOTO2,33,19,0,26.2254,31.7896,5.5642,183.6181,114.8881
3,NORMAL_2023_01_2025_03,ABLASYON_T1_CHRONOS2_vs_T0_CHRONOS2,27,17,0,12.0040,13.3520,1.3481,36.3978,5.3868
4,NORMAL_2023_01_2025_03,IS_ESIGI_T1_CHRONOS2_vs_SIFIR,27,14,0,12.0040,11.6125,-0.3915,-10.5701,-37.6154
5,NORMAL_2023_01_2025_03,KONTROL_T1_CHRONOS2_vs_TOTO2,27,14,0,12.0040,10.8192,-1.1848,-31.9894,-61.8941
6,TUM,ABLASYON_T1_CHRONOS2_vs_T0_CHRONOS2,60,42,0,19.8257,25.5800,5.7543,345.2578,269.8738
7,TUM,IS_ESIGI_T1_CHRONOS2_vs_SIFIR,60,32,0,19.8257,20.0503,0.2246,13.4749,-20.1634
8,TUM,KONTROL_T1_CHRONOS2_vs_TOTO2,60,33,0,19.8257,22.3529,2.5271,151.6287,82.8987


T1_CHRONOS2_MAE_60=19.825742
T1_CHRONOS2_DA=38/60 (63.3333%)
ABLASYON_PAIRED_WIN=42/60
SIFIR_PAIRED_WIN=32/60
KRITERLER={'ablasyon_paired_win_en_az_37': True, 'sifir_paired_win_en_az_37': False, 'sifira_gore_mae_en_az_yuzde10_iyi': False, 'yon_dogru_en_az_38': True, 'iki_rejimde_de_mae_farki_pozitif': True}
IP5_KABUL=False



## Deney kararı

Ön kayıtlı kriterlerin tamamı karşılanmadığı için `IP5_KABUL=False` sonucuna ulaşıldı. Test dönemi açılmadı. Bu notebook sonuçları yeniden üretmek ve incelemek içindir; test değerlendirmesi içermez.
